# Train Segment Classifier — Human Labels Integration

Four-stage experiment on the `feature/segment-classifier-human-labels` branch.

- **Stage 1 (baseline):** Train on auto-labels only. Report a new metric — pairwise accuracy on the human-curated pairs — to measure how often the auto-trained model agrees with humans.
- **Stage 2 (Option B):** hand-curated annotations.json overrides per-episode auto pairs at weight 1.0.
- **Stage 3 (warm-start preference elicitation):** Use the Stage 1 baseline checkpoint to drive `query.py`'s **pool-based** active learning (acquisition `H(σ(s_A − s_B)) · ‖φ_A − φ_B‖`, globally normalized features). Pool-rank every segment-pair across every episode, greedy-pick the top-K subject to a per-episode cap, label, then train.
- **Stage 4 (cold-start preference elicitation):** Same pool-based selection, no model — entropy is constant so selection collapses to **diverse pairs across the whole pool**. Bootstrap regime.

Stages 3–4 use `--budget 56 --per-episode-cap 2` so the total label budget matches Stage 2 (~56 labels) but distributed across ≥28 distinct episodes (cap-limited). The per-episode cap prevents one hard episode from swallowing the entire budget.

Stages 3–4 reuse the Stage 2 training path: `train_segment_classifier.load_annotations` auto-detects `query.py`'s list-format JSON, and the trainer now accepts multiple `(worst, clean)` pairs per episode.

**Prereq.** Push the branch first if you haven't already:
```
git push -u origin feature/segment-classifier-human-labels
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL  = "https://github.com/isa-deluis42/MAPF-GPT-DDG.git"
BRANCH    = "feature/segment-classifier-human-labels"
REPO_DIR  = "/content/MAPF-GPT-DDG"

DRIVE_ROOT = "/content/drive/MyDrive/mapf_congestion"
OUT_DIR    = f"{DRIVE_ROOT}/out/segment_classifier"

DATA_DIR        = f"{REPO_DIR}/dataset/held_out"
ANNOTATIONS     = f"{REPO_DIR}/annotations.json"
BASELINE_CKPT   = f"{OUT_DIR}/baseline.pt"
WITH_HUMAN_CKPT = f"{OUT_DIR}/with_human_labels.pt"

# Stage 3/4: elicitation JSONs are produced by running query.py locally and
# uploading to Drive. Set these to wherever you saved them.
WARMSTART_ANNOTATIONS = f"{DRIVE_ROOT}/annotation_elicitation_warmstart.json"
COLDSTART_ANNOTATIONS = f"{DRIVE_ROOT}/annotation_elicitation_coldstart.json"
WARMSTART_CKPT        = f"{OUT_DIR}/warmstart_elicitation.pt"
COLDSTART_CKPT        = f"{OUT_DIR}/coldstart_elicitation.pt"

EPOCHS          = 30
BATCH_SIZE      = 128
LR              = 3e-4
CONTEXT_SEGMENTS = 1     # 1 = this segment only; 2 = this + next as future context
BASE_CH         = 8
HOLD_OUT_EVERY  = 4      # Stages 2-4: every 4th annotation -> val
MIN_CHECKPOINT  = 0      # 0 = use all rollouts; >0 keeps only ckpt_*/ at iter >= this

DEVICE = "cuda"

import os
os.makedirs(OUT_DIR, exist_ok=True)
for p in ("REPO_DIR", "OUT_DIR", "BRANCH"):
    print(f"{p:14s} {globals()[p]}")

In [ ]:
%cd /content
!rm -rf "$REPO_DIR"
!git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git log --oneline -5

In [ ]:
# Colab already ships numpy, torch, matplotlib. Pin numpy<2 to match the rest of the project.
!python -m pip install -q --no-warn-conflicts "numpy<2" "matplotlib"

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
%%bash
# Sanity-check dataset wiring before launching the long training runs.
cd /content/MAPF-GPT-DDG
python - <<'PY'
from pathlib import Path
from train_segment_classifier import (
    load_annotations, split_annotations, SegmentPairDataset,
    is_annotated_npz, annotation_path_for,
)
from held_out_seed_set import TRAIN_MAP_SEEDS, VAL_MAP_SEEDS

ann = load_annotations('annotations.json')
tr, va = split_annotations(ann, hold_out_every=4)
print(f'annotations: {len(ann)} ({len(tr)} train + {len(va)} val)')

paths = list(Path('dataset/held_out').rglob('*.npz'))
n_ann = sum(1 for p in paths if is_annotated_npz(p))
print(f'episode files: {len(paths)} ({n_ann} annotated)')

miss = [s for s in ann if annotation_path_for(s, paths) is None]
print(f'annotations missing matching npz: {len(miss)}')

ds = SegmentPairDataset(paths, TRAIN_MAP_SEEDS, augment=False, context_segments=1, human_overrides=tr)
print(f'train pairs: {len(ds)} from {len(ds.episodes)} eps; overridden: {ds.n_overridden}')
PY


## Stage 1 — baseline (auto-labels only, human-pair accuracy as diagnostic)

We pass `--annotations` so the script reports human-pair accuracy each epoch, but force `--hold_out_every 1` so **all 26 annotations go into val and zero override training**. The model never sees human verdicts; we just measure how often it agrees with them.

Three buckets to watch for:

- **≥ 90%** — auto signal already aligned with human signal. Stage 2 will buy little.
- **60–80%** — meaningful gap. Stage 2 should help.
- **≤ 50%** — auto signal actively misleading. Stage 2 is essential.

Best epoch is selected by held-out human-pair accuracy (since `--annotations` is set). Training time on Colab GPU: roughly 30–60 min depending on the GPU.

In [ ]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {BASELINE_CKPT} \
    --annotations {ANNOTATIONS} \
    --hold_out_every 1 \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/baseline.log

## Stage 2 — Option B (human labels override training pairs)

Now `--hold_out_every 4` puts 19/26 annotations into training (each replaces its episode's auto pair list with the single human pair at weight 1.0) and 7/26 into val. We watch:

- **`human-pair acc train`** — should rise toward 1.0 (we're literally training on those pairs).
- **`human-pair acc val`** — the real test; if this rises vs. Stage 1, the human signal generalizes.
- **`val top-3 acc`** — should hold steady; if it tanks, we've over-fit to the 19 human pairs.

In [ ]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {WITH_HUMAN_CKPT} \
    --annotations {ANNOTATIONS} \
    --hold_out_every {HOLD_OUT_EVERY} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/with_human.log

## Stage 3 — warm-start preference elicitation (pool-based AL)

Use the Stage 1 baseline as the scoring model in `query.py`'s **pool-based** mode. Every (episode, segA, segB) candidate is ranked by `H(σ(s_A − s_B)) · ‖φ_A − φ_B‖` against globally-normalized features; the greedy selector picks the top-budget candidates subject to a per-episode cap.

**Run locally first** (interactive matplotlib + keyboard input):
```
python query.py dataset/held_out/filtered_npzs/annotated \
    --output annotation_elicitation_warmstart.json \
    --model-path out/segment_classifier/baseline.pt \
    --budget 56 \
    --per-episode-cap 2 \
    -r
```
Then upload `annotation_elicitation_warmstart.json` to `WARMSTART_ANNOTATIONS` on Drive.

Why a per-episode cap: without it, a single hard episode often produces 20+ near-duplicate "uncertain" pairs that swallow the whole budget and leave most episodes unlabeled — bad for generalization. Cap=2 forces coverage of at least 28 distinct episodes for budget=56.

In [ ]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {WARMSTART_CKPT} \
    --annotations {WARMSTART_ANNOTATIONS} \
    --hold_out_every {HOLD_OUT_EVERY} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/warmstart.log

## Stage 4 — cold-start preference elicitation (pool-based AL, no model)

Same pool-based selection, no model. `score_segments` returns zeros, so `p = σ(0) = 0.5` and the entropy term is constant — selection collapses to **pure feature-distance diversity sampling across the whole pool, with the per-episode cap**. Bootstrap regime: the strategy you'd use before any classifier exists.

**Run locally first:**
```
python query.py dataset/held_out/filtered_npzs/annotated \
    --output annotation_elicitation_coldstart.json \
    --budget 56 \
    --per-episode-cap 2 \
    -r
```
Upload to `COLDSTART_ANNOTATIONS` on Drive, then run the cell below.

Reading the comparison: if Stage 3 ≫ Stage 4, model uncertainty is the load-bearing piece (warm start matters); if they're close, diversity alone is doing most of the work. Either result is informative.

In [ ]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {COLDSTART_CKPT} \
    --annotations {COLDSTART_ANNOTATIONS} \
    --hold_out_every {HOLD_OUT_EVERY} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/coldstart.log

## Post-hoc: recover Stage 1 `top-1±1` from saved checkpoint

The Stage 1 run was performed before the `top-1±1` metric existed, so `baseline.log` doesn't contain it. We reload `baseline.pt` from Drive and run `evaluate_metrics` on it once — fast, no retraining, no compute on auto-pair generation. Run this cell **even if you're skipping Stage 1 today** (commented out cell 8): the checkpoint is already on Drive from the previous run.


In [ ]:
%%bash
cd /content/MAPF-GPT-DDG
python - <<'PY'
import torch
from pathlib import Path
from train_segment_classifier import (
    Segment3DCNN, evaluate_metrics, evaluate_human_pairs,
    load_annotations, split_annotations, is_annotated_npz,
)
from held_out_seed_set import VAL_MAP_SEEDS

BASELINE_PT = '/content/drive/MyDrive/mapf_congestion/out/segment_classifier/baseline.pt'
ANNOTATIONS = 'annotations.json'

ckpt = torch.load(BASELINE_PT, map_location='cuda', weights_only=False)
ctx  = ckpt.get('context_segments', 1)
model = Segment3DCNN(base_ch=ckpt['base_ch']).cuda()
model.load_state_dict(ckpt['state_dict'])

paths = list(Path('dataset/held_out').rglob('*.npz'))
n_ann = sum(1 for p in paths if is_annotated_npz(p))
print(f'episode files: {len(paths)} ({n_ann} annotated); ckpt context_segments={ctx}')

pm1, top3 = evaluate_metrics(model, paths, VAL_MAP_SEEDS, 'cuda', context_segments=ctx)
print(f'Stage 1 baseline.pt — val top-1\u00b11 = {pm1:.3f}  val top-3 = {top3:.3f}')

ann = load_annotations(ANNOTATIONS)
tr, va = split_annotations(ann, hold_out_every=4)
for name, subset in [('all', ann), ('train', tr), ('val', va)]:
    acc, n = evaluate_human_pairs(model, paths, subset, 'cuda', context_segments=ctx)
    print(f'  human-pair acc {name:<5s} = {acc:.3f}  ({n})')
PY


## Compare

In [ ]:
# Pull per-epoch metrics out of all four training logs and compare best save-criteria across stages.
import re, pathlib

# Line format:
# Epoch X | lr Y | loss Z | top-1±1 P | top-3 T | hp all/tr/val A/B/C | h-arg top1 a/t/v X/Y/Z | h-arg ±1 a/t/v P/Q/R
LINE = re.compile(
    r"Epoch\s+(\d+)\s+\|(?:\s+lr\s+[\d.eE+-]+\s+\|)?\s+loss\s+([\d.]+)"
    r"\s+\|\s+top-1.1\s+([\d.]+)\s+\|\s+top-3\s+([\d.]+)"
    r"(?:.*hp\s+all/tr/val\s+([\d.]+)/([\d.]+)/([\d.]+))?"
    r"(?:.*h-arg\s+top1\s+a/t/v\s+([\d.]+)/([\d.]+)/([\d.]+))?"
    r"(?:.*h-arg\s+.1\s+a/t/v\s+([\d.]+)/([\d.]+)/([\d.]+))?"
)

def parse_log(path):
    rows = []
    if not pathlib.Path(path).exists():
        print(f"[skip] log not found: {path}")
        return rows
    for line in pathlib.Path(path).read_text().splitlines():
        m = LINE.search(line)
        if not m: continue
        g = m.groups()
        row = {"epoch": int(g[0]), "loss": float(g[1]),
               "pm1": float(g[2]), "top3": float(g[3])}
        if g[4] is not None:
            row.update({"hp_all": float(g[4]), "hp_tr": float(g[5]), "hp_val": float(g[6])})
        if g[7] is not None:
            row.update({"t1_all": float(g[7]), "t1_tr": float(g[8]), "t1_val": float(g[9])})
        if g[10] is not None:
            row.update({"pm1h_all": float(g[10]), "pm1h_tr": float(g[11]), "pm1h_val": float(g[12])})
        rows.append(row)
    return rows

stages = [
    ("Stage 1 baseline", f"{OUT_DIR}/baseline.log",  "#666666"),
    ("Stage 2 +human",   f"{OUT_DIR}/with_human.log","#1f77b4"),
    ("Stage 3 warmstart",f"{OUT_DIR}/warmstart.log", "#2ca02c"),
    ("Stage 4 coldstart",f"{OUT_DIR}/coldstart.log", "#d62728"),
]
parsed = [(label, parse_log(path), color) for label, path, color in stages]

def best(rows, key):
    return max((r.get(key, 0.0) for r in rows), default=0.0)

print("=== Best metrics per stage (full {} epochs) ===".format(EPOCHS))
hdr = f"{'metric':28s}  " + "  ".join(f"{label:>18s}" for label, _, _ in parsed)
print(hdr); print("-" * len(hdr))
for label, key in [
    ("val top-1±1 (DDG-aligned)", "pm1"),
    ("val top-3",                       "top3"),
    ("hp pairwise val",                 "hp_val"),
    ("hp pairwise all",                 "hp_all"),
    ("h-arg top1 val",                  "t1_val"),
    ("h-arg top1 all",                  "t1_all"),
    ("h-arg ±1 val",              "pm1h_val"),
    ("h-arg ±1 all",              "pm1h_all"),
]:
    cells = "  ".join(f"{best(rows,key):>18.3f}" for _, rows, _ in parsed)
    print(f"{label:28s}  {cells}")

print()
print("=== Saved checkpoints ===")
import os
expected = []
for stem in ["with_human_labels", "warmstart_elicitation", "coldstart_elicitation"]:
    expected += [
        f"{stem}.pt",
        f"{stem}.argmax_pm1.pt",
        f"{stem}.human_argmax.pt",
        f"{stem}.human_argmax_top1.pt",
    ]
for name in expected:
    p = f"{OUT_DIR}/{name}"
    print(f"  {name:42s}  {'exists' if os.path.exists(p) else '<missing>'}")

# Plot comparison: hp pairwise val + human-argmax top1 val + DDG top-1±1.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for label, rows, color in parsed:
    if not rows: continue
    ep = [r["epoch"] for r in rows]
    if rows and "hp_val" in rows[0]:
        axes[0].plot(ep, [r["hp_val"] for r in rows], label=label, color=color)
    if rows and "t1_val" in rows[0]:
        axes[1].plot(ep, [r["t1_val"] for r in rows], label=label, color=color)
    axes[2].plot(ep, [r["pm1"] for r in rows], label=label, color=color)
axes[0].axhline(0.5,   color="red", linestyle=":", alpha=0.4, label="chance")
axes[1].axhline(0.354, color="red", linestyle=":", alpha=0.4, label="random")
axes[0].set_ylabel("HP Pairwise val");           axes[0].set_xlabel("Epoch"); axes[0].grid(True); axes[0].legend(); axes[0].set_ylim(0,1)
axes[1].set_ylabel("HP Argmax top-1 val");       axes[1].set_xlabel("Epoch"); axes[1].grid(True); axes[1].legend(); axes[1].set_ylim(0,1)
axes[2].set_ylabel("Val top-1±1 (auto)");   axes[2].set_xlabel("Epoch"); axes[2].grid(True); axes[2].legend(); axes[2].set_ylim(0,1)
fig.tight_layout()
plt.savefig(f"{OUT_DIR}/comparison.png", dpi=150)
plt.show()
